# 📦 Otimização de Estoque e Previsão de Demanda
### Brazilian E-Commerce Public Dataset by Olist

---

Esse projeto analisa mais de 100 mil pedidos reais do e-commerce brasileiro (Olist, 2016–2018) pra responder uma pergunta prática: **como usar dados históricos de vendas pra tomar decisões melhores de estoque?**

O que vamos fazer:

1. **Conhecer os dados** – entender a estrutura e qualidade da base
2. **Limpar os dados** – tratar nulos, datas e anomalias
3. **Realizar EDA + Curva ABC** – identificar os produtos mais rentáveis
4. **Prever a demanda** – usar Machine Learning (Prophet) para projetar os próximos 60 dias
5. **Gerar um relatório gerencial** – conclusões e plano de ação

> **Dataset:** [Brazilian E-Commerce Public Dataset by Olist — Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

---

## ⚙️ 0. Configuração do Ambiente

Execute a célula abaixo **uma única vez** para instalar todas as bibliotecas necessárias.

In [ ]:
# Instalar dependências (execute apenas uma vez)
import subprocess, sys

libs = [
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'prophet', 'kaggle', 'openpyxl'
]

for lib in libs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])

print('✅ Todas as bibliotecas instaladas com sucesso!')

## 📥 1. Ingestão e Conhecimento dos Dados

### Como baixar o Dataset Olist

**Opção A – Manual (mais simples):**
1. Acesse [kaggle.com/datasets/olistbr/brazilian-ecommerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
2. Clique em **Download** e extraia os arquivos `.csv` na pasta `data/` deste projeto

**Opção B – Via API Kaggle (automatizado):**
```bash
# Configure suas credenciais em ~/.kaggle/kaggle.json
kaggle datasets download -d olistbr/brazilian-ecommerce --unzip -p data/
```

Após o download, a pasta `data/` deve conter:
- `olist_orders_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_products_dataset.csv`
- `olist_product_category_name_translation.csv`
- `olist_customers_dataset.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# ── Estilo visual ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#2a2a4a',
    'axes.labelcolor':  '#e0e0ff',
    'xtick.color':      '#a0a0c0',
    'ytick.color':      '#a0a0c0',
    'text.color':       '#e0e0ff',
    'grid.color':       '#2a2a4a',
    'grid.linestyle':   '--',
    'figure.dpi':       120,
    'font.family':      'DejaVu Sans',
})

PALETTE = ['#7c3aed', '#a855f7', '#c084fc', '#e879f9', '#f0abfc']
ACCENT  = '#7c3aed'
GOLD    = '#f59e0b'

print('✅ Bibliotecas carregadas e estilo visual configurado.')

In [ ]:
DATA_DIR = 'data'

# ── Carregar tabelas ───────────────────────────────────────────────────────────
print('📂 Carregando datasets...')

orders      = pd.read_csv(f'{DATA_DIR}/olist_orders_dataset.csv')
order_items = pd.read_csv(f'{DATA_DIR}/olist_order_items_dataset.csv')
products    = pd.read_csv(f'{DATA_DIR}/olist_products_dataset.csv')
os.makedirs("output", exist_ok=True)
# Auto-detectar nome do arquivo de categorias (varia entre versoes do Kaggle)
_candidates = [
    f"{DATA_DIR}/product_category_name_translation.csv",
    f"{DATA_DIR}/olist_product_category_name_translation.csv",
]
_cat_file = next((fn for fn in _candidates if os.path.exists(fn)), None)
if _cat_file is None:
    raise FileNotFoundError("Arquivo de categorias nao encontrado em data/")
category_pt = pd.read_csv(_cat_file)
print(f"  Categorias carregadas de: {_cat_file}")
customers   = pd.read_csv(f'{DATA_DIR}/olist_customers_dataset.csv')

print(f'  ✔ Pedidos:         {orders.shape[0]:,} linhas')
print(f'  ✔ Itens de pedido: {order_items.shape[0]:,} linhas')
print(f'  ✔ Produtos:        {products.shape[0]:,} linhas')
print(f'  ✔ Clientes:        {customers.shape[0]:,} linhas')
print('\n✅ Todos os datasets carregados!')

In [ ]:
# ── Visão geral da tabela principal ───────────────────────────────────────────
print('=' * 60)
print('📋 Primeiras linhas — orders')
print('=' * 60)
display(orders.head())

print('\n' + '=' * 60)
print('📋 Primeiras linhas — order_items')
print('=' * 60)
display(order_items.head())

In [ ]:
# ── Tipos de dados e valores nulos ────────────────────────────────────────────
print('=' * 60)
print('🔍 Informações — orders')
print('=' * 60)
orders.info()

print('\n' + '=' * 60)
print('📊 Estatísticas — order_items')
print('=' * 60)
display(order_items.describe().round(2))

## 🧹 2. Limpeza de Dados (Data Cleaning)

Dados reais sempre vêm "sujos". Nesta etapa vamos:
- Converter colunas de data para o tipo `datetime`
- Remover pedidos cancelados ou com status inválido
- Tratar valores nulos em categorias de produtos
- Criar a tabela analítica consolidada (merge)
- Filtrar anomalias de preço e quantidade

In [ ]:
# ── 2.1 Converter colunas de data ──────────────────────────────────────────────
DATE_COLS = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in DATE_COLS:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print(f'✅ Colunas de data convertidas: {DATE_COLS}')
print(f'   Pedidos com data nula: {orders["order_purchase_timestamp"].isna().sum()}')

In [ ]:
# ── 2.2 Filtrar apenas pedidos entregues ───────────────────────────────────────
print(f'Status de pedidos:\n{orders["order_status"].value_counts()}\n')

orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f'✅ Pedidos entregues: {orders_delivered.shape[0]:,} '
      f'({orders_delivered.shape[0]/orders.shape[0]*100:.1f}% do total)')

In [ ]:
# ── 2.3 Traduzir categorias para português ─────────────────────────────────────
products_clean = products.merge(category_pt, on='product_category_name', how='left')
products_clean['category'] = (
    products_clean['product_category_name_english']
    .fillna(products_clean['product_category_name'])
    .fillna('sem_categoria')
    .str.replace('_', ' ')
    .str.title()
)

print(f'✅ Categorias únicas: {products_clean["category"].nunique()}')
print(f'   Produtos sem categoria: {(products_clean["category"] == "Sem Categoria").sum()}')

In [ ]:
# ── 2.4 Criar tabela analítica consolidada ─────────────────────────────────────
df = (
    order_items
    .merge(orders_delivered[['order_id', 'order_purchase_timestamp']], on='order_id', how='inner')
    .merge(products_clean[['product_id', 'category']], on='product_id', how='left')
)

# Renomear e criar features de tempo
df.rename(columns={'order_purchase_timestamp': 'data_compra'}, inplace=True)
df['ano_mes']      = df['data_compra'].dt.to_period('M')
df['mes']          = df['data_compra'].dt.month
df['dia_semana']   = df['data_compra'].dt.dayofweek
df['nome_dia']     = df['data_compra'].dt.day_name()
df['receita']      = df['price'] + df['freight_value']

# ── 2.5 Remover anomalias ──────────────────────────────────────────────────────
antes = len(df)
df = df[df['price'] > 0]          # preço deve ser positivo
df = df[df['freight_value'] >= 0]  # frete não pode ser negativo
df.dropna(subset=['data_compra'], inplace=True)

print(f'✅ Tabela analítica criada!')
print(f'   Linhas antes da limpeza: {antes:,}')
print(f'   Linhas após a limpeza:   {len(df):,}')
print(f'   Período coberto:         {df["data_compra"].min().date()} → {df["data_compra"].max().date()}')
display(df.head())

## 📊 3. Análise Exploratória (EDA) + Curva ABC

### 3.1 Evolução do Faturamento ao Longo do Tempo (Sazonalidade)

In [ ]:
# ── Faturamento mensal ─────────────────────────────────────────────────────────
faturamento_mensal = (
    df.groupby('ano_mes')['receita']
    .sum()
    .reset_index()
)
faturamento_mensal['ano_mes_dt'] = faturamento_mensal['ano_mes'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(
    faturamento_mensal['ano_mes_dt'],
    faturamento_mensal['receita'],
    alpha=0.25, color=ACCENT
)
ax.plot(
    faturamento_mensal['ano_mes_dt'],
    faturamento_mensal['receita'],
    color=ACCENT, linewidth=2.5, marker='o', markersize=5
)

# Destacar pico máximo
idx_max = faturamento_mensal['receita'].idxmax()
ax.annotate(
    f'Pico\nR$ {faturamento_mensal.loc[idx_max, "receita"]/1e6:.1f}M',
    xy=(faturamento_mensal.loc[idx_max, 'ano_mes_dt'], faturamento_mensal.loc[idx_max, 'receita']),
    xytext=(30, -40), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color=GOLD),
    color=GOLD, fontsize=9, fontweight='bold'
)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x/1e6:.1f}M'))
ax.set_title('📈 Faturamento Mensal — Olist E-Commerce', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Mês')
ax.set_ylabel('Faturamento Total')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('output/01_faturamento_mensal.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo em output/01_faturamento_mensal.png')

### 3.2 Vendas por Dia da Semana

In [ ]:
dias_ordem  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dias_pt     = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']

vendas_dia = (
    df.groupby('nome_dia')['receita']
    .sum()
    .reindex(dias_ordem)
    .reset_index()
)
vendas_dia['dia_pt'] = dias_pt

fig, ax = plt.subplots(figsize=(9, 4))

colors = [GOLD if v == vendas_dia['receita'].max() else ACCENT
          for v in vendas_dia['receita']]

bars = ax.bar(vendas_dia['dia_pt'], vendas_dia['receita'], color=colors,
              width=0.6, edgecolor='none')

for bar, val in zip(bars, vendas_dia['receita']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'R$ {val/1e6:.1f}M', ha='center', va='bottom',
            fontsize=8, color='#e0e0ff')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x/1e6:.1f}M'))
ax.set_title('📅 Faturamento por Dia da Semana', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Dia da Semana')
ax.set_ylabel('Faturamento Total')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('output/02_vendas_por_dia.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Curva ABC — Análise de Pareto por Categoria

In [ ]:
# ── Curva ABC ─────────────────────────────────────────────────────────────────
receita_categoria = (
    df.groupby('category')['receita']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

total_receita = receita_categoria['receita'].sum()
receita_categoria['% acumulado'] = receita_categoria['receita'].cumsum() / total_receita * 100

def classificar_abc(pct):
    if pct <= 80:  return 'A'
    elif pct <= 95: return 'B'
    return 'C'

receita_categoria['Classe'] = receita_categoria['% acumulado'].apply(classificar_abc)

# Resumo
resumo_abc = receita_categoria.groupby('Classe').agg(
    Categorias=('category', 'count'),
    Receita_Total=('receita', 'sum'),
    Participação_Pct=('receita', lambda x: x.sum()/total_receita*100)
).round(2)

print('=' * 55)
print('📊 RESUMO DA CURVA ABC')
print('=' * 55)
display(resumo_abc)
print(f'\nTotal de categorias: {len(receita_categoria)}')
print(f'Classe A gera {resumo_abc.loc["A","Participação_Pct"]:.1f}% da receita com '
      f'apenas {resumo_abc.loc["A","Categorias"]} categorias')

In [ ]:
# ── Visualização da Curva ABC ─────────────────────────────────────────────────
top_n  = 20
df_plot = receita_categoria.head(top_n).copy()

color_map = {'A': '#7c3aed', 'B': '#f59e0b', 'C': '#6b7280'}
bar_colors = df_plot['Classe'].map(color_map)

fig, ax1 = plt.subplots(figsize=(16, 7))

bars = ax1.bar(df_plot['category'], df_plot['receita'],
               color=bar_colors, edgecolor='none', width=0.7)

ax1.set_ylabel('Receita Total (R$)', color='#e0e0ff')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x/1e6:.1f}M'))
ax1.set_xticklabels(df_plot['category'], rotation=45, ha='right', fontsize=8)

# Linha de % acumulado
ax2 = ax1.twinx()
ax2.plot(df_plot['category'], df_plot['% acumulado'],
         color='#e879f9', linewidth=2, marker='D', markersize=5, zorder=5)
ax2.axhline(80,  color='#7c3aed', linestyle='--', linewidth=1.2, alpha=0.8, label='80% — Classe A')
ax2.axhline(95,  color='#f59e0b', linestyle='--', linewidth=1.2, alpha=0.8, label='95% — Classe B')
ax2.set_ylabel('% Acumulado', color='#e879f9')
ax2.set_ylim(0, 115)
ax2.legend(loc='center right', fontsize=8)

# Legenda de classes
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#7c3aed', label='Classe A — 80% receita'),
    Patch(facecolor='#f59e0b', label='Classe B — próximos 15%'),
    Patch(facecolor='#6b7280', label='Classe C — últimos 5%'),
]
ax1.legend(handles=legend_elements, loc='upper right', fontsize=8)

ax1.set_title('📦 Curva ABC — Pareto de Receita por Categoria (Top 20)',
              fontsize=14, fontweight='bold', pad=15)
ax1.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('output/03_curva_abc.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo em output/03_curva_abc.png')

In [ ]:
# ── Top 10 categorias Classe A ─────────────────────────────────────────────────
top_a = receita_categoria[receita_categoria['Classe'] == 'A'].head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_a['category'][::-1], top_a['receita'][::-1],
               color=ACCENT, edgecolor='none')

for bar, val in zip(bars, top_a['receita'][::-1]):
    ax.text(bar.get_width() + total_receita*0.001,
            bar.get_y() + bar.get_height()/2,
            f'R$ {val/1e6:.2f}M', va='center', fontsize=9, color='#e0e0ff')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x/1e6:.1f}M'))
ax.set_title('🏆 Top 10 Categorias — Classe A (Prioridade Máxima de Estoque)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Receita Total')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('output/04_top_categorias_a.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 4. Previsão de Demanda com Prophet (Machine Learning)

Utilizamos o **Prophet**, biblioteca de séries temporais desenvolvida pela Meta/Facebook, para prever a demanda dos próximos **60 dias** na categoria com maior receita (Classe A).

### Por que Prophet?
- Lida nativamente com sazonalidade (semanal, anual)
- Robusto a dados faltantes
- Gera intervalos de confiança automaticamente
- Simples de interpretar

In [ ]:
from prophet import Prophet

# ── Selecionar a categoria Classe A mais vendida ───────────────────────────────
categoria_alvo = receita_categoria[receita_categoria['Classe'] == 'A'].iloc[0]['category']
print(f'🎯 Categoria selecionada para previsão: "{categoria_alvo}"')

# ── Preparar série temporal diária ────────────────────────────────────────────
df_cat = df[df['category'] == categoria_alvo].copy()
df_cat['data'] = df_cat['data_compra'].dt.date

serie_diaria = (
    df_cat.groupby('data')['receita']
    .sum()
    .reset_index()
    .rename(columns={'data': 'ds', 'receita': 'y'})
)
serie_diaria['ds'] = pd.to_datetime(serie_diaria['ds'])

print(f'\n📅 Período de dados: {serie_diaria["ds"].min().date()} → {serie_diaria["ds"].max().date()}')
print(f'   Dias com dados:   {len(serie_diaria)}')
print(f'   Receita média/dia: R$ {serie_diaria["y"].mean():,.2f}')
display(serie_diaria.tail())

In [ ]:
# ── Treinar o modelo Prophet ───────────────────────────────────────────────────
modelo = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',  # melhor para dados de e-commerce
    interval_width=0.95                 # 95% de confiança
)

modelo.fit(serie_diaria)
print('✅ Modelo Prophet treinado com sucesso!')

# ── Gerar previsão para 60 dias ───────────────────────────────────────────────
DIAS_PREVISAO = 60
futuro     = modelo.make_future_dataframe(periods=DIAS_PREVISAO)
previsao   = modelo.predict(futuro)

print(f'\n📆 Previsão gerada para {DIAS_PREVISAO} dias à frente')
display(previsao[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10).round(2))

In [ ]:
# ── Visualização da Previsão ───────────────────────────────────────────────────
historico = previsao[previsao['ds'] <= serie_diaria['ds'].max()]
futuro_df = previsao[previsao['ds'] >  serie_diaria['ds'].max()]

fig, ax = plt.subplots(figsize=(15, 6))

# Dados reais
ax.plot(serie_diaria['ds'], serie_diaria['y'],
        color='#a0a0c0', linewidth=1, alpha=0.7, label='Vendas reais')

# Fitted (histórico)
ax.plot(historico['ds'], historico['yhat'],
        color=ACCENT, linewidth=1.8, label='Modelo ajustado')

# Previsão futura
ax.plot(futuro_df['ds'], futuro_df['yhat'],
        color=GOLD, linewidth=2.5, linestyle='--', label=f'Previsão ({DIAS_PREVISAO} dias)')
ax.fill_between(futuro_df['ds'],
                futuro_df['yhat_lower'], futuro_df['yhat_upper'],
                color=GOLD, alpha=0.2, label='Intervalo de confiança 95%')

# Linha divisória
ax.axvline(serie_diaria['ds'].max(), color='#e879f9',
           linestyle=':', linewidth=1.5, alpha=0.8)
ax.text(serie_diaria['ds'].max(), ax.get_ylim()[1]*0.95,
        ' Hoje', color='#e879f9', fontsize=9)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
ax.set_title(f'🔮 Previsão de Demanda — {categoria_alvo} (próximos {DIAS_PREVISAO} dias)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Data')
ax.set_ylabel('Receita Diária (R$)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/05_previsao_prophet.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo em output/05_previsao_prophet.png')

In [ ]:
# ── Componentes da Sazonalidade ────────────────────────────────────────────────
fig = modelo.plot_components(previsao)
fig.set_facecolor('#0f0f1a')
for ax in fig.axes:
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#a0a0c0')
    ax.title.set_color('#e0e0ff')
    ax.yaxis.label.set_color('#a0a0c0')
    ax.xaxis.label.set_color('#a0a0c0')
    for spine in ax.spines.values():
        spine.set_edgecolor('#2a2a4a')
    for line in ax.get_lines():
        line.set_color(ACCENT)

plt.suptitle('📉 Componentes da Sazonalidade', fontsize=13, fontweight='bold',
             y=1.02, color='#e0e0ff')
plt.tight_layout()
plt.savefig('output/06_componentes_sazonalidade.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Métricas de previsão ───────────────────────────────────────────────────────
receita_prevista_total = futuro_df['yhat'].sum()
receita_prevista_media_dia = futuro_df['yhat'].mean()
receita_historica_media_dia = serie_diaria['y'].mean()

variacao_pct = (receita_prevista_media_dia - receita_historica_media_dia) / receita_historica_media_dia * 100

print('=' * 55)
print('📊 MÉTRICAS DE PREVISÃO')
print('=' * 55)
print(f'  Categoria analisada:         {categoria_alvo}')
print(f'  Período previsto:            {DIAS_PREVISAO} dias')
print(f'  Receita média histórica/dia: R$ {receita_historica_media_dia:,.2f}')
print(f'  Receita média prevista/dia:  R$ {receita_prevista_media_dia:,.2f}')
print(f'  Variação esperada:           {variacao_pct:+.1f}%')
print(f'  Receita total prevista:      R$ {receita_prevista_total:,.2f}')
print('=' * 55)

sinal = '📈' if variacao_pct > 0 else '📉'
print(f'\n{sinal} Conclusão: A categoria "{categoria_alvo}" apresenta variação '
      f'de {variacao_pct:+.1f}% na demanda para os próximos {DIAS_PREVISAO} dias.')

## 📝 5. Conclusão e Plano de Ação Gerencial

---

### 🎯 Sumário Executivo

Esta análise processou a base completa de pedidos do e-commerce Olist, cobrindo o período de 2016 a 2018. A seguir estão as principais descobertas e recomendações estratégicas.

---

### 📦 Descoberta 1 — Lei de Pareto (Curva ABC)

> **Insight:** Um número pequeno de categorias de produtos é responsável por 80% de toda a receita da operação (Classe A).

**Plano de Ação:**
- ✅ **Prioridade máxima de reabastecimento** deve ser exclusiva para produtos Classe A. Rupturas de estoque nessas categorias são inaceitáveis e causam perda direta de receita.
- ⚠️ **Revisão dos produtos Classe C**: São muitos SKUs diferentes com vendas ínfimas. Recomenda-se avaliar descontinuação ou negociação de menores quantidades com fornecedores para liberar capital de giro.
- 🔄 **Negociação com fornecedores**: Concentre poder de barganha nas categorias Classe A, que movimentam maior volume e justificam melhores condições comerciais.

---

### 📅 Descoberta 2 — Sazonalidade e Picos de Demanda

> **Insight:** Há picos claros de vendas identificados no gráfico de faturamento mensal, com períodos de baixa previsíveis.

**Plano de Ação:**
- 📦 **Pré-compra estratégica**: 45 dias antes dos picos identificados, acionar compras adicionais para produtos Classe A. O custo de estoque extra é menor que a perda por ruptura.
- 🏷️ **Promoções nos vales**: Nos meses de menor demanda, criar campanhas para escoar estoque e manter o giro saudável.
- 📋 **Calendário de compras**: Criar um calendário anual alinhando as ordens de compra com os padrões sazonais identificados.

---

### 🔮 Descoberta 3 — Previsão de Demanda (60 dias)

> **Insight:** O modelo Prophet identificou tendências e sazonalidades que permitem antecipar o volume de vendas com intervalo de confiança de 95%.

**Plano de Ação:**
- 📬 **Acionar o setor de compras agora**: Com base na previsão, definir a quantidade mínima de estoque de segurança para as próximas 8 semanas.
- 📊 **Atualização mensal do modelo**: Retreinar o Prophet mensalmente com dados novos para manter a precisão das previsões.
- ⚠️ **Plano de contingência**: O intervalo de confiança superior da previsão deve ser utilizado como cenário pessimista de reposição — garanta estoque para o pior caso.

---

### 🏁 Próximos Passos

| # | Ação | Prazo | Responsável |
|---|------|-------|-------------|
| 1 | Revisar policy de estoque mínimo para categorias Classe A | 7 dias | Logística |
| 2 | Emitir PO de reposição para os próximos 60 dias (baseado no modelo) | 15 dias | Compras |
| 3 | Iniciar avaliação de descontinuação de SKUs Classe C | 30 dias | Produto + Compras |
| 4 | Criar dashboard em Power BI / Streamlit com previsão atualizada | 45 dias | BI |
| 5 | Reunião de revisão com liderança apresentando este relatório | 7 dias | Gestão |

---

*Projeto desenvolvido por Gustavo Guedes Mantellis · Python + Prophet + Pandas · Dataset: Olist / Kaggle*

In [ ]:
# ── Exportar tabela ABC completa ───────────────────────────────────────────────
receita_categoria.to_csv('output/curva_abc_completa.csv', index=False)
print('✅ Tabela Curva ABC exportada: output/curva_abc_completa.csv')

# ── Exportar previsão ──────────────────────────────────────────────────────────
previsao[['ds','yhat','yhat_lower','yhat_upper']].to_csv(
    'output/previsao_demanda.csv', index=False
)
print('✅ Previsão exportada: output/previsao_demanda.csv')

print('\n🎉 Análise completa! Todos os outputs foram salvos na pasta output/')